# PySpark ETL
This notebook focuses on performing ETL (Extract, Transform, Load) operations on the data ingested from the FPL API. We will use PySpark to clean, transform, and prepare the data for model training.

## 1. Include required libraries

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import * # Import all functions
from pyspark.sql.window import Window
import os
from datetime import datetime

## 2. Initialize Spark Session
We initialize a Spark session, which is the entry point to any Spark functionality.

In [5]:
spark = SparkSession.builder.appName("gameweek-prophet-etl").getOrCreate()

## 3. Read Data
We read the CSV files generated by earlier steps into Spark DataFrames.

In [6]:
# data_dir = "../data/raw/fpl_api/20250314_085615/"
# elements_df = spark.read.csv(f"{data_dir}elements.csv", header=True, inferSchema=True)
# teams_df = spark.read.csv(f"{data_dir}teams.csv", header=True, inferSchema=True)
# events_df = spark.read.csv(f"{data_dir}events.csv", header=True, inferSchema=True)
# player_history_df = spark.read.csv(f"{data_dir}player_history.csv", header=True, inferSchema=True)
# team_fixtures_df = spark.read.csv(f"{data_dir}team_fixtures.csv", header=True, inferSchema=True)

In [7]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, BooleanType, TimestampType

# Define the schema for the FPL historical data
schema_merged_gw_2022_2023 = StructType([
    StructField("name", StringType(), True),
    StructField("position", StringType(), True),
    StructField("team", StringType(), True),
    StructField("xP", FloatType(), True),
    StructField("assists", IntegerType(), True),
    StructField("bonus", IntegerType(), True),
    StructField("bps", IntegerType(), True),
    StructField("clean_sheets", IntegerType(), True),
    StructField("creativity", FloatType(), True),
    StructField("element", IntegerType(), True),
    StructField("expected_assists", FloatType(), True),
    StructField("expected_goal_involvements", FloatType(), True),
    StructField("expected_goals", FloatType(), True),
    StructField("expected_goals_conceded", FloatType(), True),
    StructField("fixture", IntegerType(), True),
    StructField("goals_conceded", IntegerType(), True),
    StructField("goals_scored", IntegerType(), True),
    StructField("ict_index", FloatType(), True),
    StructField("influence", FloatType(), True),
    StructField("kickoff_time", TimestampType(), True),
    StructField("minutes", IntegerType(), True),
    StructField("opponent_team", IntegerType(), True),
    StructField("own_goals", IntegerType(), True),
    StructField("penalties_missed", IntegerType(), True),
    StructField("penalties_saved", IntegerType(), True),
    StructField("red_cards", IntegerType(), True),
    StructField("round", IntegerType(), True),
    StructField("saves", IntegerType(), True),
    StructField("selected", IntegerType(), True),
    StructField("starts", IntegerType(), True),
    StructField("team_a_score", IntegerType(), True),
    StructField("team_h_score", IntegerType(), True),
    StructField("threat", FloatType(), True),
    StructField("total_points", IntegerType(), True),
    StructField("transfers_balance", IntegerType(), True),
    StructField("transfers_in", IntegerType(), True),
    StructField("transfers_out", IntegerType(), True),
    StructField("value", FloatType(), True),
    StructField("was_home", BooleanType(), True),
    StructField("yellow_cards", IntegerType(), True),
    StructField("GW", IntegerType(), True)
])

schema_merged_gw_2024 = StructType([
    StructField("name", StringType(), True),
    StructField("position", StringType(), True),
    StructField("team", StringType(), True),
    StructField("xP", FloatType(), True),
    StructField("assists", IntegerType(), True),
    StructField("bonus", IntegerType(), True),
    StructField("bps", IntegerType(), True),
    StructField("clean_sheets", IntegerType(), True),
    StructField("creativity", FloatType(), True),
    StructField("element", IntegerType(), True),
    StructField("expected_assists", FloatType(), True),
    StructField("expected_goal_involvements", FloatType(), True),
    StructField("expected_goals", FloatType(), True),
    StructField("expected_goals_conceded", FloatType(), True),
    StructField("fixture", IntegerType(), True),
    StructField("goals_conceded", IntegerType(), True),
    StructField("goals_scored", IntegerType(), True),
    StructField("ict_index", FloatType(), True),
    StructField("influence", FloatType(), True),
    StructField("kickoff_time", TimestampType(), True),
    StructField("minutes", IntegerType(), True),
    StructField("modified", BooleanType(), True),  # Added for 2024_25_merged_gw.csv
    StructField("opponent_team", IntegerType(), True),
    StructField("own_goals", IntegerType(), True),
    StructField("penalties_missed", IntegerType(), True),
    StructField("penalties_saved", IntegerType(), True),
    StructField("red_cards", IntegerType(), True),
    StructField("round", IntegerType(), True),
    StructField("saves", IntegerType(), True),
    StructField("selected", IntegerType(), True),
    StructField("starts", IntegerType(), True),
    StructField("team_a_score", IntegerType(), True),
    StructField("team_h_score", IntegerType(), True),
    StructField("threat", FloatType(), True),
    StructField("total_points", IntegerType(), True),
    StructField("transfers_balance", IntegerType(), True),
    StructField("transfers_in", IntegerType(), True),
    StructField("transfers_out", IntegerType(), True),
    StructField("value", FloatType(), True),
    StructField("was_home", BooleanType(), True),
    StructField("yellow_cards", IntegerType(), True),
    StructField("GW", IntegerType(), True)
])

data_dir = "../data/raw/fpl_historical/20250313_225438/"
input_files_schema = {
    os.path.join(data_dir, "2024_25_merged_gw_fixed.csv"): schema_merged_gw_2024,
    os.path.join(data_dir, "2023_24_merged_gw.csv"): schema_merged_gw_2022_2023,
    os.path.join(data_dir, "2022_23_merged_gw.csv"): schema_merged_gw_2022_2023,
}

# Read each file with the common schema
dfs = [
    spark.read.csv(file, header=True, schema=schema)
    for file, schema in input_files_schema.items()
]

# Combine all DataFrames into one
fpl_history_df = dfs[0]
for df in dfs:
    fpl_history_df = fpl_history_df.unionByName(df, allowMissingColumns=True)

# Show the combined DataFrame
fpl_history_df.show()

# for df in dfs:
#     df.show()

25/03/22 17:44:14 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------------------+--------+--------------+---+-------+-----+---+------------+----------+-------+----------------+--------------------------+--------------+-----------------------+-------+--------------+------------+---------+---------+-------------------+-------+--------+-------------+---------+----------------+---------------+---------+-----+-----+--------+------+------------+------------+------+------------+-----------------+------------+-------------+-----+--------+------------+---+
|                name|position|          team| xP|assists|bonus|bps|clean_sheets|creativity|element|expected_assists|expected_goal_involvements|expected_goals|expected_goals_conceded|fixture|goals_conceded|goals_scored|ict_index|influence|       kickoff_time|minutes|modified|opponent_team|own_goals|penalties_missed|penalties_saved|red_cards|round|saves|selected|starts|team_a_score|team_h_score|threat|total_points|transfers_balance|transfers_in|transfers_out|value|was_home|yellow_cards| GW|
+---------

## 4. Data Cleaning
We'll handle data cleaning tasks such as dealing with missing values and standardizing team/player names. But first let's pick which features data points we are interested in so we don't waste effort on cleaning data that is not required.

### Dimensions
* Player name
* Team name
* Opponent team name
* Position
* Kickoff time
* Game week
* FPL season

### Metrics
* Goals scored
* Goals conceded
* Assists
* Expected assists
* Expected goal involvements
* Expected goals
* Expected goals conceded
* Penalties scored
* Penalties missed
* Penalties saved
* Saves
* BPS (Bonus Points System)
* Points
* Minutes
* Yellow cards
* Red cards
* Own goals
* Starts
* Was home
* Influence
* Creativity
* Threat
* ICT index
* Clean sheets

In [8]:
filter_cols = [
    "name",
    "team",
    "position",
    "kickoff_time",
    "GW",
    "goals_scored",
    "goals_conceded",
    "assists",
    "expected_assists",
    "expected_goal_involvements",
    "expected_goals",
    "expected_goals_conceded",
    "total_points",
    "penalties_missed",
    "penalties_saved",
    "saves",
    "bps",
    "minutes",
    "yellow_cards",
    "red_cards",
    "own_goals",
    "starts",
    "was_home",
    "influence",
    "creativity",
    "threat",
    "ict_index",
    "clean_sheets"
]
    
# Filter the columns in the DataFrame
filtered_df = fpl_history_df.select(filter_cols)

# Show the filtered DataFrame
filtered_df.show(5)

+--------------------+--------------+--------+-------------------+---+------------+--------------+-------+----------------+--------------------------+--------------+-----------------------+------------+----------------+---------------+-----+---+-------+------------+---------+---------+------+--------+---------+----------+------+---------+------------+
|                name|          team|position|       kickoff_time| GW|goals_scored|goals_conceded|assists|expected_assists|expected_goal_involvements|expected_goals|expected_goals_conceded|total_points|penalties_missed|penalties_saved|saves|bps|minutes|yellow_cards|red_cards|own_goals|starts|was_home|influence|creativity|threat|ict_index|clean_sheets|
+--------------------+--------------+--------+-------------------+---+------------+--------------+-------+----------------+--------------------------+--------------+-----------------------+------------+----------------+---------------+-----+---+-------+------------+---------+---------+------

In [9]:
# from pyspark.sql.types import IntegerType, FloatType, StringType, TimestampType

# # Define the desired data types for each column
# column_types = {
#     "name": StringType(),
#     "team": StringType(),
#     "position": StringType(),
#     "kickoff_time": TimestampType(),
#     "GW": IntegerType(),
#     "goals_scored": IntegerType(),
#     "goals_conceded": IntegerType(),
#     "assists": IntegerType(),
#     "expected_assists": FloatType(),
#     "expected_goal_involvements": FloatType(),
#     "expected_goals": FloatType(),
#     "expected_goals_conceded": FloatType(),
#     "total_points": IntegerType(),
#     "penalties_missed": IntegerType(),
#     "penalties_saved": IntegerType(),
#     "saves": IntegerType(),
#     "bps": IntegerType(),
#     "minutes": IntegerType(),
#     "yellow_cards": IntegerType(),
#     "red_cards": IntegerType(),
#     "own_goals": IntegerType(),
#     "starts": IntegerType(),
#     "was_home": IntegerType(),
#     "influence": FloatType(),
#     "creativity": FloatType(),
#     "threat": FloatType(),
#     "ict_index": FloatType(),
#     "clean_sheets": IntegerType()
# }

# # Cast each column to its specified data type
# for col_name, col_type in column_types.items():
#     filtered_df = filtered_df.withColumn(col_name, filtered_df[col_name].cast(col_type))

# # Show the DataFrame with updated data types
# filtered_df.printSchema()
# filtered_df.show(5)

In [10]:
from pyspark.sql.functions import col, sum

# Count missing values for each column
missing_values = filtered_df.select(
    [(sum(col(c).isNull().cast("int")).alias(c)) for c in filtered_df.columns]
)

# Show the count of missing values
missing_values.show()

# Calculate total missing values across all columns
total_missing = missing_values.selectExpr("stack({0}, {1}) as (column, missing_count)".format(
    len(filtered_df.columns),
    ", ".join([f"'{c}', {c}" for c in filtered_df.columns])
)).agg(sum("missing_count")).collect()[0][0]

print(f"Total missing values in DataFrame: {total_missing}")


+----+----+--------+------------+---+------------+--------------+-------+----------------+--------------------------+--------------+-----------------------+------------+----------------+---------------+-----+---+-------+------------+---------+---------+------+--------+---------+----------+------+---------+------------+
|name|team|position|kickoff_time| GW|goals_scored|goals_conceded|assists|expected_assists|expected_goal_involvements|expected_goals|expected_goals_conceded|total_points|penalties_missed|penalties_saved|saves|bps|minutes|yellow_cards|red_cards|own_goals|starts|was_home|influence|creativity|threat|ict_index|clean_sheets|
+----+----+--------+------------+---+------------+--------------+-------+----------------+--------------------------+--------------+-----------------------+------------+----------------+---------------+-----+---+-------+------------+---------+---------+------+--------+---------+----------+------+---------+------------+
|   0|   0|       0|           0|  0|

In [ ]:
# from pyspark.sql.functions import col

# # Filter rows with missing values in any column
# rows_with_missing_values = filtered_df.filter(
#     " OR ".join([f"{c} IS NULL" for c in filtered_df.columns])
# )

# # Show the records with missing values
# rows_with_missing_values.show()

+--------------------+--------------+--------+-------------------+---+------------+--------------+-------+----------------+--------------------------+--------------+-----------------------+------------+----------------+---------------+-----+---+-------+------------+---------+---------+------+--------+---------+----------+------+---------+------------+
|                name|          team|position|       kickoff_time| GW|goals_scored|goals_conceded|assists|expected_assists|expected_goal_involvements|expected_goals|expected_goals_conceded|total_points|penalties_missed|penalties_saved|saves|bps|minutes|yellow_cards|red_cards|own_goals|starts|was_home|influence|creativity|threat|ict_index|clean_sheets|
+--------------------+--------------+--------+-------------------+---+------------+--------------+-------+----------------+--------------------------+--------------+-----------------------+------------+----------------+---------------+-----+---+-------+------------+---------+---------+------

## 5. Feature Engineering
We'll engineer rolling window features to capture recent and longer-term performance trends. For example, we'll calculate rolling averages for player statistics like total points, goals scored, etc.

### Features
* [team/player] Goals scored [last 4/last 5-16]
* [team/player] Goals conceded [last 4/last 5-16]
* [team/player] Assists [last 4/last 5-16]
* [team/player] Expected assists [last 4/last 5-16]
* [team/player] Expected goal involvements [last 4/last 5-16]
* [team/player] Expected goals [last 4/last 5-16]
* [team/player] Expected goals conceded [last 4/last 5-16]
* [team/player] Penalties scored [last 4/last 5-16]
* [team/player] Penalties missed [last 4/last 5-16]
* [team/player] Penalties saved [last 4/last 5-16]
* [team/player] Saves [last 4/last 5-16]
* [team/player] BPS (Bonus Points System) [last 4/last 5-16]
* [team/player] Points [last 4/last 5-16]
* [team/player] Minutes [last 4/last 5-16]
* [team/player] Yellow cards [last 4/last 5-16]
* [team/player] Red cards [last 4/last 5-16]
* [team/player] Own goals [last 4/last 5-16]
* [team/player] Starts [last 4/last 5-16]
* [team/player] Was home [last 4/last 5-16]
* [team/player] Influence [last 4/last 5-16]
* [team/player] Creativity [last 4/last 5-16]
* [team/player] Threat [last 4/last 5-16]
* [team/player] ICT index [last 4/last 5-16]
* [team/player] Clean sheets [last 4/last 5-16]

In [20]:
# Define a window specification for recent performance (e.g., last 4 weeks)
# We define a window to calculate rolling metrics over the last 4 weeks.
recent_window = Window.partitionBy("name","position").orderBy("kickoff_time").rowsBetween(Window.currentRow - 4, Window.currentRow - 1)

# Define a window specification for longer-term performance (e.g., last 4 to 12 weeks)
# We define a window to calculate rolling metrics over the last 4 to 12 weeks.
long_term_window = Window.partitionBy("name","position").orderBy("kickoff_time").rowsBetween(Window.currentRow - 12, Window.currentRow - 5)

# List of metrics to get rolling metrics for
metrics = [
    "goals_scored",
    "goals_conceded",
    "assists",
    "expected_assists",
    "expected_goal_involvements",
    "expected_goals",
    "expected_goals_conceded",
    "total_points",
    "penalties_missed",
    "penalties_saved",
    "saves",
    "bps",
    "minutes",
    "yellow_cards",
    "red_cards",
    "own_goals",
    "starts",
    "influence",
    "creativity",
    "threat",
    "ict_index",
    "clean_sheets"
]

# Make a copy of filtered_df to calculate features
features_df = filtered_df

# Calculate rolling metrics with a check for sufficient rows
for metric in metrics:
    # Calculate recent rolling average with row count check
    features_df = features_df.withColumn(
        f"recent_{metric}",
        when(
            count(lit(1)).over(recent_window) >= 4,  # Check if we have at least 4 rows
            sum(metric).over(recent_window)
        ).otherwise(None)  # Return None if not enough rows
    )
    
    # Calculate long-term rolling average with row count check
    features_df = features_df.withColumn(
        f"long_term_{metric}",
        when(
            count(lit(1)).over(long_term_window) >= 8,  # Check for at least 8 rows
            sum(metric).over(long_term_window)
        ).otherwise(None)  # Return None if not enough rows
    )

# Show features dataframe
features_df.show()

+---------------+--------+--------+-------------------+---+------------+--------------+-------+----------------+--------------------------+--------------+-----------------------+------------+----------------+---------------+-----+---+-------+------------+---------+---------+------+--------+---------+----------+------+---------+------------+-------------------+----------------------+---------------------+------------------------+--------------+-----------------+-----------------------+--------------------------+---------------------------------+------------------------------------+---------------------+------------------------+------------------------------+---------------------------------+-------------------+----------------------+-----------------------+--------------------------+----------------------+-------------------------+------------+---------------+----------+-------------+--------------+-----------------+-------------------+----------------------+----------------+-----------

In [33]:
from pyspark.sql.functions import col, sum

# Count missing values for each column
missing_values = features_df.select(
    [format_number(avg(col(c).isNull().cast("int")), 2).alias(c) for c in features_df.columns]
)

# Show the percentage of missing values
missing_values.show()

# Filter rows with missing values in any column
rows_with_missing_values = features_df.filter(
    " OR ".join([f"{c} IS NULL" for c in features_df.columns])
).count()

# Calculate the total number of rows
total_rows = features_df.count()

# Calculate the percentage of rows with at least one missing value
percentage_missing = rows_with_missing_values / total_rows

# Print the formatted string
print(f"Percentage of records with at least one missing value: {percentage_missing:.2f}")

# Drop rows with missing values
features_df = features_df.filter(
    " OR ".join([f"{c} IS NULL" for c in features_df.columns])
)

print(f"Number of records after dropping rows with missing values: {features_df.count()}")

+----+----+--------+------------+----+------------+--------------+-------+----------------+--------------------------+--------------+-----------------------+------------+----------------+---------------+-----+----+-------+------------+---------+---------+------+--------+---------+----------+------+---------+------------+-------------------+----------------------+---------------------+------------------------+--------------+-----------------+-----------------------+--------------------------+---------------------------------+------------------------------------+---------------------+------------------------+------------------------------+---------------------------------+-------------------+----------------------+-----------------------+--------------------------+----------------------+-------------------------+------------+---------------+----------+-------------+--------------+-----------------+-------------------+----------------------+----------------+-------------------+-----------

Percentage of records with at least one missing value: 1.00
Number of records after dropping rows with missing values: 16456


## 6. Write Data
Finally, we'll write the transformed data to CSV files for use in the subsequent model training notebook.

In [36]:
# Define output directory
# We define the directory where the processed data will be saved.
data_source = "features"
current_datetime = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = f"../data/processed/{data_source}/{current_datetime}"
filename = f"features.parquet"
filepath = os.path.join(output_dir, filename)

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Create the directory if it doesn't exist
# We create the directory if it doesn't exist.
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Write the transformed DataFrames to Parquet files
# We save the processed data as parquet files.
features_df.write.parquet(f"{filepath}", mode="overwrite")

print(f"ETL process complete. Transformed data saved to {filepath}")

ETL process complete. Transformed data saved to ../data/processed/features/20250322_184141/features.parquet
